# PhosphoAtlas Excel EDA

Minimal exploratory notebook for inspecting the PhosphoAtlas kinase-substrate Excel file before using it for background kinase selection.

In [3]:
from pathlib import Path
import pandas as pd

In [4]:
excel_path = Path("../data/phosphoAtlas_data/2024_PhosphoAtlas 2.0_updated KSP network_withHTKAMconnections.xlsx")
xls = pd.ExcelFile(excel_path)

print("Excel path:", excel_path)
print("Sheet names:", xls.sheet_names)

Excel path: ../data/phosphoAtlas_data/2024_PhosphoAtlas 2.0_updated KSP network_withHTKAMconnections.xlsx
Sheet names: ['2024_PhosphoAtlas 2.0_updated K']


In [5]:
sheet_name = xls.sheet_names[0]
df = pd.read_excel(excel_path, sheet_name=sheet_name)

print("Sheet loaded:", sheet_name)
print("Shape:", df.shape)

Sheet loaded: 2024_PhosphoAtlas 2.0_updated K
Shape: (14043, 10)


In [6]:
df.head(10)

,KINASE GENE,KINASE common name,KIN_ACC_ID,SUBSTRATE common name,SUB_GENE_ID,SUB_ACC_ID,SUBSTRATE GENE,SUB_MOD_RSD,SITE_GRP_ID,SITE_+/-7_AA
0,AAK1,AAK1,Q2M2I8,AP2M1,1173.0,Q96CW1,AP2M1,T156,448782.0,SQItsQVtGQIGWRR
1,AAK1,AAK1,Q2M2I8,NUMB,8650.0,P49757,NUMB,T102,3333403.0,LRVVDEKtKDLIVDQ
2,ABL1,Abl,P00519,Abi-1,10006.0,Q8IZP0,ABI1,Y213,449523.0,PPtVPNDyMtsPARL
3,ABL1,Abl,P00519,Abl,25.0,P00519,ABL1,Y393,450298.0,RLMtGDtytAHAGAk
4,ABL1,Abl,P00519,Abl iso2,25.0,P00519-2,ABL1,Y412,450298.0,RLMTGDtyTAHAGAK
5,ABL1,Abl,P00519,Arg,27.0,P42684,ABL2,Y261,455657.0,GLVTTLHyPAPKCNK
6,ABL1,Abl,P00519,AHSA1,10598.0,O95433,AHSA1,Y223,3914778.0,LtsPEELyRVFTTQE
7,ABL1,Abl,P00519,ANXA1,301.0,P04083,ANXA1,Y21,447974.0,IENEEQEyVQtVkss
8,ABL1,Abl,P00519,RhoGDI beta,397.0,P52566,ARHGDIB,Y130,4113623.0,LkYVQHtyRTGVkVD
9,ABL1,Abl,P00519,RhoGDI beta,397.0,P52566,ARHGDIB,Y24,448978.0,ELdskLNykPPPQks


In [7]:
df.columns.tolist()

['KINASE GENE',
 'KINASE common name',
 'KIN_ACC_ID',
 'SUBSTRATE common name',
 'SUB_GENE_ID',
 'SUB_ACC_ID',
 'SUBSTRATE GENE',
 'SUB_MOD_RSD',
 'SITE_GRP_ID',
 'SITE_+/-7_AA']

In [8]:
df.dtypes

KINASE GENE                  str
KINASE common name           str
KIN_ACC_ID                   str
SUBSTRATE common name     object
SUB_GENE_ID              float64
SUB_ACC_ID                   str
SUBSTRATE GENE            object
SUB_MOD_RSD                  str
SITE_GRP_ID              float64
SITE_+/-7_AA                 str
dtype: object

In [9]:
summary = {
    "n_rows": len(df),
    "n_columns": df.shape[1],
    "n_unique_kinase_genes": df["KINASE GENE"].nunique(dropna=True),
    "n_unique_kinase_accessions": df["KIN_ACC_ID"].nunique(dropna=True),
    "n_unique_substrate_genes": df["SUBSTRATE GENE"].nunique(dropna=True),
    "n_unique_substrate_accessions": df["SUB_ACC_ID"].nunique(dropna=True),
}
pd.Series(summary)

n_rows                           14043
n_columns                           10
n_unique_kinase_genes              406
n_unique_kinase_accessions         418
n_unique_substrate_genes          2837
n_unique_substrate_accessions     2958
dtype: int64

In [10]:
duplicate_summary = {
    "full_row_duplicates": int(df.duplicated().sum()),
    "duplicate_kinase_accessions": int(df["KIN_ACC_ID"].duplicated().sum()),
    "duplicate_substrate_accessions": int(df["SUB_ACC_ID"].duplicated().sum()),
    "duplicate_sites_within_kinase_substrate": int(
        df.duplicated(subset=["KIN_ACC_ID", "SUB_ACC_ID", "SUB_MOD_RSD", "SITE_GRP_ID"]).sum()
    ),
}
pd.Series(duplicate_summary)

full_row_duplicates                            0
duplicate_kinase_accessions                13624
duplicate_substrate_accessions             11084
duplicate_sites_within_kinase_substrate      207
dtype: int64

In [11]:
df["KIN_ACC_ID"].value_counts().head(20)

KIN_ACC_ID
P06493    661
P17612    607
P68400    571
P12931    554
P17252    469
P28482    443
P24941    420
P27361    385
P49841    364
P31749    356
P53350    303
Q13315    291
P00519    221
Q16539    221
Q96GD4    215
O14757    196
Q13131    162
P45983    161
Q00535    157
P06241    146
Name: count, dtype: int64

In [12]:
df[["KINASE GENE", "KINASE common name", "KIN_ACC_ID"]].drop_duplicates().head(20)

,KINASE GENE,KINASE common name,KIN_ACC_ID
0,AAK1,AAK1,Q2M2I8
2,ABL1,Abl,P00519
223,ABL2,Arg,P42684
245,ACVR1B,ALK4,P36896
247,ACVRL1,ALK1,P37023
251,ADCK5,ADCK5,Q3MIX3
252,AKT1,Akt1,P31749
608,AKT2,Akt2,P31751
681,AKT3,Akt3,Q9Y243
697,ALK,ALK,Q9UM73


In [13]:
df[["KINASE GENE", "KINASE common name", "KIN_ACC_ID"]].drop_duplicates().sort_values(["KINASE GENE", "KIN_ACC_ID"]).reset_index(drop=True)

,KINASE GENE,KINASE common name,KIN_ACC_ID
0,AAK1,AAK1,Q2M2I8
1,ABL1,Abl,P00519
2,ABL2,Arg,P42684
3,ACVR1B,ALK4,P36896
4,ACVRL1,ALK1,P37023
...,...,...,...
500,WNK4,WNK4,Q96J92
501,YES1,Yes,P07947
502,YES1,NaN,NaN
503,ZAP70,ZAP70,P43403
